# 08 - Fetch the raw weather
This file fetches the raw weather data from the API, and saves it as one compressed JSON Lines file (one line per tournament) in raw/open_meteo/archive_v2/, instead of one JSON file per tournament


## Install the pinned libraries

In [ ]:

# Read from a git tag rather than from main, so a later re-run installs exactly the same versions.
CODE_VERSION = "code-v1"      # the git tag of the code being run: create it on GitHub before running
REPO = "katiedevane/tennis-project"
REQUIREMENTS_URL = f"https://raw.githubusercontent.com/{REPO}/{CODE_VERSION}/requirements.txt"
!pip install -q -r $REQUIREMENTS_URL

# Shared provenance helpers (code version, environment, index of data versions), from the same tag
PROVENANCE_URL = f"https://raw.githubusercontent.com/{REPO}/{CODE_VERSION}/provenance.py"
!wget -q -O provenance.py $PROVENANCE_URL

## Sign in and load settings from Colab Secrets
Switch on Notebook access for both secrets in this notebook.

In [ ]:
import os
from google.colab import auth, userdata

auth.authenticate_user()

PROJECT_ID = userdata.get("TENNIS_PROJECT_ID")
BUCKET_NAME = userdata.get("TENNIS_BUCKET_NAME")
REGION = "EU"
DATASETS = ["tennis_raw", "tennis_clean", "tennis_features"]
os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID

assert PROJECT_ID and BUCKET_NAME, "A secret is empty: check its name and Notebook access"
print("Settings loaded from Colab Secrets.")

## Record the code version and set the random seed
The git tag, commit, Python environment and random seed of this run are added to every manifest this notebook writes (see provenance.py).
`SEED` is the one project-wide seed: `start_run` applies it, and it should be passed wherever a function takes a seed (e.g. `random_state=SEED`).

In [ ]:
from provenance import start_run, register, table_version, sha256, utc_stamp, SEED

NOTEBOOK = "08_weather_raw"
RUN_INFO = start_run(f"https://github.com/{REPO}", CODE_VERSION, NOTEBOOK)

## Load the edition list

In [ ]:
import pandas as pd
from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT_ID, location=REGION)
editions = bq.query(
    f"SELECT * FROM `{PROJECT_ID}.tennis_clean.tournament_editions` ORDER BY tourney_id",
    job_config=bigquery.QueryJobConfig(maximum_bytes_billed=1_000_000_000),
).to_dataframe()
for c in ["start_date", "end_date"]:
    editions[c] = pd.to_datetime(editions[c])
print(len(editions), "editions loaded")

## The fetch function
Each response is saved exactly as returned, with the request and fetch time stored beside it.
If a response is already stored (in a packed file, or as a single file from an earlier run) it is only skipped when it was fetched with exactly the parameters now wanted.
The model is pinned to ERA5.
The first run of this cell reads the 872 single files from earlier runs, which takes a few minutes.

In [ ]:
import gzip, json, time, hashlib
import requests
from datetime import datetime, timezone
from google.cloud import storage

gcs = storage.Client(project=PROJECT_ID)
bucket = gcs.bucket(BUCKET_NAME)

ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
MODEL = "era5"
HOURLY = ["temperature_2m", "relative_humidity_2m", "dew_point_2m", "surface_pressure",
          "precipitation", "wind_speed_10m", "wind_gusts_10m", "wind_direction_10m", "cloud_cover"]
RAW_PREFIX = f"raw/open_meteo/archive_v2/model={MODEL}"
PACKED_PREFIX = f"{RAW_PREFIX}/packed/"
CHECKPOINT = f"{PACKED_PREFIX}_checkpoint.jsonl.gz"     # saved during the fetch, so a crash loses little

def pack(records):
    """One gzip JSON Lines file, one line per record, in a fixed order (mtime=0: same records -> same bytes)."""
    lines = [json.dumps(r, sort_keys=True) for r in sorted(records, key=lambda r: r["tourney_id"])]
    return gzip.compress(("\n".join(lines) + "\n").encode(), mtime=0)

def unpack(data):
    return [json.loads(line) for line in gzip.decompress(data).splitlines()]

# Responses already stored: single files from earlier runs first, then packed files (newer ones win)
stored = {}
for b in gcs.list_blobs(BUCKET_NAME, prefix=f"{RAW_PREFIX}/"):
    if b.name.endswith(".json"):
        rec = json.loads(b.download_as_bytes())
        stored[rec["tourney_id"]] = rec
for b in sorted(gcs.list_blobs(BUCKET_NAME, prefix=PACKED_PREFIX), key=lambda b: b.time_created):
    for rec in unpack(b.download_as_bytes()):
        stored[rec["tourney_id"]] = rec
print("responses already stored:", len(stored))

def fetch_edition(row, model=MODEL, store=True):
    """Fetch one edition's weather. With store=True: skip if already stored, else keep it for packing."""
    params = {
        "latitude": round(float(row.latitude), 4), "longitude": round(float(row.longitude), 4),
        "start_date": row.start_date.strftime("%Y-%m-%d"), "end_date": row.end_date.strftime("%Y-%m-%d"),
        "hourly": ",".join(HOURLY), "timezone": "auto",
    }
    if model:
        params["models"] = model

    if store and row.tourney_id in stored:
        if stored[row.tourney_id]["request"]["params"] != params:
            raise RuntimeError(f"{row.tourney_id}: stored response was fetched with different parameters")
        return "already stored", None

    for attempt in range(5):                       # retry on busy or temporary server errors
        r = requests.get(ARCHIVE_URL, params=params, timeout=60)
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(2 * 2 ** attempt)           # wait 2, 4, 8, 16, 32 seconds
            continue
        r.raise_for_status()
        break
    else:
        raise RuntimeError(f"Gave up on {row.tourney_id} (HTTP {r.status_code})")

    record = {
        "tourney_id": row.tourney_id,
        "request": {"url": ARCHIVE_URL, "params": params},
        "fetched_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "response": r.json(),                      # exactly as returned
    }
    if store:
        stored[row.tourney_id] = record
    return "fetched", record

## Test that the model setting works (nothing is saved if different model used)
identical should be 'FALSE'

In [ ]:
row = next(editions[editions["start_date"].dt.strftime("%Y-%m") == "2024-06"].itertuples())
_, pinned = fetch_edition(row, model="era5", store=False)
time.sleep(1)
_, default = fetch_edition(row, model=None, store=False)

a = pinned["response"]["hourly"]["temperature_2m"]
b = default["response"]["hourly"]["temperature_2m"]
print(row.place, row.start_date.date())
print("era5    first 6 hours:", a[:6])
print("default first 6 hours:", b[:6])
print("identical:", a == b)

## Fetch every edition
If `failed` is not empty, run this cell once more.

In [ ]:
failed, counts = [], {"fetched": 0, "already stored": 0}
for i, row in enumerate(editions.itertuples(), 1):
    try:
        status, _ = fetch_edition(row)
        counts[status] += 1
        if status == "fetched":
            time.sleep(0.5)                        # about 2 requests a second at most
            if counts["fetched"] % 100 == 0:       # checkpoint, so a crash does not lose the fetched responses
                bucket.blob(CHECKPOINT).upload_from_string(pack(stored.values()), content_type="application/gzip")
    except Exception as e:
        failed.append((row.tourney_id, repr(e)[:150]))
    if i % 100 == 0:
        print(i, counts, "failed:", len(failed))

print("Done:", counts, "| failed:", len(failed))
failed[:5]

## Pack the responses into one file
The responses for all editions are written to one compressed file. The file name holds its checksum, so a file is never overwritten and an unchanged set is not stored twice.

Single files from earlier runs (`raw/open_meteo/archive_v2/model=era5/*.json`) are no longer used and can be deleted from the bucket once the checks below pass.

In [ ]:
records = [stored[t] for t in sorted(set(editions["tourney_id"])) if t in stored]
packed = pack(records)
packed_sha = hashlib.sha256(packed).hexdigest()
WEATHER_FILE = f"{PACKED_PREFIX}weather_{MODEL}_{packed_sha[:12]}.jsonl.gz"

blob = bucket.blob(WEATHER_FILE)
if blob.exists():
    print(WEATHER_FILE, ": already stored, identical")
else:
    blob.upload_from_string(packed, content_type="application/gzip", if_generation_match=0)
    print(WEATHER_FILE, ": saved")
print(len(records), "editions |", round(len(packed) / 1e6, 1), "MB compressed")

checkpoint = bucket.blob(CHECKPOINT)
if checkpoint.exists():
    checkpoint.delete()
    print("checkpoint removed")

## Check what is stored
Read back from the packed file.


In [ ]:
data = bucket.blob(WEATHER_FILE).download_as_bytes()
assert hashlib.sha256(data).hexdigest() == packed_sha, "The stored file differs from what was packed"

entries = []
for rec in unpack(data):
    line = json.dumps(rec, sort_keys=True)
    hourly = rec["response"]["hourly"]
    p = rec["request"]["params"]
    entries.append({
        "tourney_id": rec["tourney_id"], "bytes": len(line),
        "sha256": hashlib.sha256(line.encode()).hexdigest(),
        "start_date": p["start_date"], "end_date": p["end_date"],
        "n_hours": len(hourly["time"]),
        "null_values": sum(v is None for k, vals in hourly.items() if k != "time" for v in vals),
        "fetched_at_utc": rec["fetched_at_utc"],
    })

wanted = editions[["tourney_id", "start_date", "end_date"]].copy()
wanted["start_date"] = wanted["start_date"].dt.strftime("%Y-%m-%d")
wanted["end_date"] = wanted["end_date"].dt.strftime("%Y-%m-%d")
check = wanted.merge(pd.DataFrame(entries), on="tourney_id", how="left", suffixes=("_wanted", "_stored"))
check["expected_hours"] = ((pd.to_datetime(check["end_date_wanted"]) - pd.to_datetime(check["start_date_wanted"])).dt.days + 1) * 24

print("records stored:", len(entries), "| editions with no record:", int(check["sha256"].isna().sum()))
print("records with a different window than wanted:",
      int(((check["start_date_stored"] != check["start_date_wanted"]) | (check["end_date_stored"] != check["end_date_wanted"])).sum()))
print("records whose hour count differs from the window by more than 2:", int(((check["n_hours"] - check["expected_hours"]).abs() > 2).sum()))
print("total null values:", int(check["null_values"].sum()))
print("total size (MB):", round(check["bytes"].sum() / 1e6, 1), "| compressed file (MB):", round(len(data) / 1e6, 1))

## Look at one stored record (timezone, units, first hours)

In [ ]:
rec = next(r for r in unpack(data) if r["tourney_id"] == editions.iloc[0].tourney_id)
resp = rec["response"]
print({k: resp.get(k) for k in ["latitude", "longitude", "elevation", "timezone", "hourly_units"]})
pd.DataFrame(resp["hourly"]).head(6)

## Save the manifest

In [ ]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
manifest_path = f"manifests/open_meteo_archive_v2_{MODEL}_{stamp}.json"
file_entry = {"file": WEATHER_FILE, "bytes": len(data), "sha256": packed_sha, "records": len(entries)}
bucket.blob(manifest_path).upload_from_string(
    json.dumps({"endpoint": ARCHIVE_URL, "model": MODEL, "hourly_variables": HOURLY,
                "window_rule": "start = tourney_date; end = latest possible match day + 1 (tennis_clean.tournament_editions)",
                "files": [file_entry], "records": entries, **RUN_INFO}, indent=1),
    content_type="application/json")
print("Saved", manifest_path, "with", len(entries), "records")

register(bucket, "raw/open_meteo_weather", packed_sha[:12], outputs=[file_entry],
         parents=[f"{ARCHIVE_URL} (model {MODEL})", table_version(bq, f"{PROJECT_ID}.tennis_clean.tournament_editions")],
         manifest_path=manifest_path)

#Compare every single weather file with its record in the packed file

In [ ]:
# Compare every single weather file with its record in the packed file (content, not bytes)
packed = {r["tourney_id"]: r for r in unpack(bucket.blob(WEATHER_FILE).download_as_bytes())}
singles = [b for b in gcs.list_blobs(BUCKET_NAME, prefix=f"{RAW_PREFIX}/") if b.name.endswith(".json")]
diff = [b.name for b in singles
        if json.loads(b.download_as_bytes()) != packed.get(json.loads(b.download_as_bytes())["tourney_id"])]
print("single files:", len(singles), "| packed records:", len(packed), "| differing:", len(diff))
assert not diff, diff[:5]